In [4]:
import subprocess
import pandas
import io

pandas.options.mode.copy_on_write = True

result = subprocess.run(["cloc", "--include-lang=Scala", "--by-file", "--csv", "../"], capture_output=True)
cloc = pandas.read_csv(io.BytesIO(result.stdout))
cloc = cloc[["filename", "code"]]
cloc.drop(cloc.tail(1).index, inplace=True)

cloc

,filename,code
0,../sturdy-wasm/swam/core/src/swam/traversal/Tr...,2700
1,../sturdy-wasm/swam/text/src/swam/text/Resolve...,1232
2,../sturdy-wasm/swam/core/src/swam/validation/S...,1197
3,../sturdy-wasm/swam/core/src/swam/binary/InstC...,1028
4,../sturdy-wasm/swam/core/src/swam/traversal/Tr...,1026
...,...,...
501,../sturdy-tip/src/main/scala/sturdy/language/t...,0
502,../sturdy-tip/src/test/scala/sturdy/language/t...,0
503,../sturdy-tip/src/test/scala/sturdy/language/t...,0
504,../sturdy-tip/src/test/scala/sturdy/language/t...,0


In [87]:
def sturdy_tip(re: str):
    return cloc.filename.str.contains('sturdy-tip/src/main/scala/sturdy/language/tip/'+re, regex= True, na=False)

def sturdy_wasm(re: str):
    return cloc.filename.str.contains('sturdy-wasm/src/main/scala/sturdy/language/wasm/'+re, regex= True, na=False)

def sturdy_apron(re: str):
    return cloc.filename.str.contains('sturdy-apron/src/main/scala/sturdy/'+re, regex= True, na=False)

def sturdy_core(re: str):
    return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)

def toTex(res):
    total_loc = res['code'].sum()
    print('\\def\\sumCode{'+str(res['code'].sum())+' loc\\xspace}')
    print('\\def\\sumReuse{'+str(res[res['reuse'] == True]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumNoReuse{'+str(res[res['reuse'] == False]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageIndependent{'+str(res[res['language-independent'] == True]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageSpecific{'+str(res[res['language-independent'] == False]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageIndependentReuse{'+str(res[(res['language-independent'] == True) & (res['reuse'] == True)]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageIndependentNoReuse{'+str(res[(res['language-independent'] == True) & (res['reuse'] == False)]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageSpecificReuse{'+str(res[(res['language-independent'] == False) & (res['reuse'] == True)]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumLanguageSpecificNoReuse{'+str(res[(res['language-independent'] == False) & (res['reuse'] == False)]['code'].sum())+' loc\\xspace}')
    print('\\def\\sumReuseOrLanguageIndependent{'+str(res[(res['language-independent'] == True) | (res['reuse'] == True)]['code'].sum())+' loc\\xspace}')

    print('\\def\\percentCode{'+"{:.1f}".format(res['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentReuse{'+"{:.1f}".format(res[res['reuse'] == True]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentNoReuse{'+"{:.1f}".format(res[res['reuse'] == False]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageIndependent{'+"{:.1f}".format(res[res['language-independent'] == True]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageSpecific{'+"{:.1f}".format(res[res['language-independent'] == False]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageIndependentReuse{'+"{:.1f}".format(res[(res['language-independent'] == True) & (res['reuse'] == True)]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageIndependentNoReuse{'+"{:.1f}".format(res[(res['language-independent'] == True) & (res['reuse'] == False)]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageSpecificReuse{'+"{:.1f}".format(res[(res['language-independent'] == False) & (res['reuse'] == True)]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentLanguageSpecificNoReuse{'+"{:.1f}".format(res[(res['language-independent'] == False) & (res['reuse'] == False)]['code'].sum()/total_loc*100)+'\\%\\xspace}')
    print('\\def\\percentReuseOrLanguageIndependent{'+"{:.1f}".format(res[(res['language-independent'] == True) | (res['reuse'] == True)]['code'].sum()/total_loc*100)+'\\%\\xspace}')

In [82]:
fixpoint_language_independent = cloc[
    sturdy_core('fix/Combinators.scala') |
    sturdy_core('fix/Contextual.scala') |
    sturdy_core('fix/Fixpoint.scala') |
    sturdy_core('fix/Stack.scala') |
    sturdy_core('fix/StackedStates.scala') |
    sturdy_core('fix/State.scala') |
    sturdy_core('fix/context/Sensitivity.scala') |
    sturdy_core('fix/iter/.*')
]

common = pandas.concat([fixpoint_language_independent, cloc[
    sturdy_core('data/.*') |
    sturdy_core('effect/Effect.*') |
    sturdy_core('effect/ComputationJoiner.scala') |
    sturdy_core('effect/TrySturdy.scala')
]])

In [88]:
### Tip

# Language-specific, reuse
tip1 = cloc[
    sturdy_tip('generic/') |
    sturdy_tip('Interpreter.scala') |
    sturdy_tip('abstractions/Functions.scala') |
    sturdy_tip('abstractions/Records.scala') |
    sturdy_tip('abstractions/Control.scala') |
    sturdy_tip('abstractions/Fix.scala')
]
tip1["language-independent"] = False
tip1["reuse"] = True


# Language-specific, no reuse
tip2 = cloc[
    sturdy_wasm('analyses/RelationalAnalysis.scala')
]
tip2["language-independent"] = False
tip2["reuse"] = False

# Language-independent, reuse
tip3 = pandas.concat([common, cloc[
    sturdy_core('effect/allocation/(Allocator|AAllocatorFromContext).scala') |
    sturdy_core('effect/callframe/(CallFrame|DecidableCallFrame).scala') |
    sturdy_core('effect/failure/(Failure|AFallible|CollectedFailures).scala') |
    sturdy_core('effect/print/(Print|PrintBound).scala') |
    sturdy_core('effect/userinput/(A)?UserInput.scala') |
    sturdy_core('values/booleans/(Lifted)?Boolean(Branching|Ops).scala') |
    sturdy_core('values/integer/(Lifted)?IntegerOps.scala') |
    sturdy_core('values/ordering/(Lifted)?(Ordering|Eq)Ops.scala') |
    sturdy_core('values/references/(AbstractAddr|AbstractReference|(Lifted)?ReferenceOps).scala') |
    sturdy_core('values/types/BaseType.scala') |
    sturdy_core('values/Join.scala') |
    sturdy_core('values/Topped.scala')
]])
tip3["language-independent"] = True
tip3["reuse"] = True

# Language-independent, no reuse
tip4 = cloc[
    sturdy_core('RecencyAddr.*') |
    sturdy_core('RecencyStore.*') |
    sturdy_apron('apron/.*') |
    sturdy_apron('effect/callframe/.*') |
    sturdy_apron('effect/store/RelationalStore.scala') |
    sturdy_apron('effect/store/RelationalStore.scala') |
    sturdy_apron('values/booleans/RelationalBoolBranching.scala') |
    sturdy_apron('values/booleans/RelationalBooleanOps.scala') |
    sturdy_apron('values/integer/RelationalIntegerOps.scala') |
    sturdy_apron('values/ordering/RelationalOrderingOps.scala')
]
tip4["language-independent"] = True
tip4["reuse"] = False

res = pandas.concat([tip1, tip2, tip3, tip4])

toTex(res)
print('\\def\\sumLanguageIndependentReuseFixpoint{'+str(fixpoint_language_independent['code'].sum())+' loc}')
print('\\def\\sumLanguageSpecificReuseFixpoint{'+str(cloc[sturdy_tip('abstractions/Fix.scala')]['code'].sum())+' loc}')
print('\\def\\sumLanguageSpecificNoReuseFixpoint{9 loc}')

\def\sumCode{6375 loc\xspace}
\def\sumReuse{2642 loc\xspace}
\def\sumNoReuse{3733 loc\xspace}
\def\sumLanguageIndependent{5215 loc\xspace}
\def\sumLanguageSpecific{1160 loc\xspace}
\def\sumLanguageIndependentReuse{2369 loc\xspace}
\def\sumLanguageIndependentNoReuse{2846 loc\xspace}
\def\sumLanguageSpecificReuse{273 loc\xspace}
\def\sumLanguageSpecificNoReuse{887 loc\xspace}
\def\sumReuseOrLanguageIndependent{5488 loc\xspace}
\def\percentCode{100.0\%\xspace}
\def\percentReuse{41.4\%\xspace}
\def\percentNoReuse{58.6\%\xspace}
\def\percentLanguageIndependent{81.8\%\xspace}
\def\percentLanguageSpecific{18.2\%\xspace}
\def\percentLanguageIndependentReuse{37.2\%\xspace}
\def\percentLanguageIndependentNoReuse{44.6\%\xspace}
\def\percentLanguageSpecificReuse{4.3\%\xspace}
\def\percentLanguageSpecificNoReuse{13.9\%\xspace}
\def\percentReuseOrLanguageIndependent{86.1\%\xspace}
\def\sumLanguageIndependentReuseFixpoint{717 loc}
\def\sumLanguageSpecificReuseFixpoint{73 loc}
\def\sumLanguageSpecific

/tmp/ipykernel_26979/2015922807.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)


In [89]:
### Wasm
# Language-specific, reuse
wasm1 = cloc[
    sturdy_wasm('generic/') |
    sturdy_wasm('Interpreter.scala') |
    sturdy_wasm('Properties.scala') |
    sturdy_wasm('abstractions/ExceptionByTarget.scala') |
    sturdy_wasm('abstractions/ControlFlow.scala')
]
wasm1["language-independent"] = False
wasm1["reuse"] = True


# Language-specific, no reuse
wasm2 = cloc[
    sturdy_wasm('analyses/RelationalAnalysis.scala') |
    sturdy_wasm('abstractions/Relational(Addresses|I32Values|Memory|Types|Values).scala')
]
wasm2["language-independent"] = False
wasm2["reuse"] = False

# Language-independent, reuse
wasm3 = pandas.concat([common, tip4, cloc[
    sturdy_core('effect/allocation/(Allocator|AAllocatorFromContext).scala') |
    sturdy_core('effect/callframe/(CallFrame|DecidableCallFrame).scala') |
    sturdy_core('effect/bytememory/(Memory|AlignedMemory).scala') |
    sturdy_core('effect/operandstack/(Decidable)?OperandStack.scala') |
    sturdy_core('effect/symboltable/(Decidable|Interval)?SymbolTable.scala') |
    sturdy_core('effect/except/(Joined)?Except.scala') |
    sturdy_core('effect/failure/(Failure|AFallible|CollectedFailures).scala') |
    sturdy_core('values/addresses/AddressOps.scala') |
    sturdy_core('values/booleans/(Lifted)?Boolean(Branching|Ops).scala') |
    sturdy_core('values/config/.*') |
    sturdy_core('values/convert/(Lifted|Transitive)?Convert.scala') |
    sturdy_core('values/exceptional/Exceptional(ByTarget)?.scala') |
    sturdy_core('values/functions/(Powerset|Lifted)?FunctionOps.scala') |
    sturdy_core('values/floating/(Lifted)?FloatOps.scala') |
    sturdy_core('values/integer/(Lifted)?IntegerOps.scala') |
    sturdy_core('values/simd/((Lifted|Concrete)?SIMDOps|SIMDType).scala') |
    sturdy_core('values/ordering/(Lifted)?(Unsigned)?(Ordering|Eq)Ops.scala') |
    sturdy_core('values/references/(AbstractAddr|AbstractReference|(Lifted)?ReferenceOps).scala') |
    sturdy_core('values/types/BaseType.scala') |
    sturdy_core('values/Join.scala') |
    sturdy_core('values/Topped.scala')
]])
wasm3["language-independent"] = True
wasm3["reuse"] = True

# Language-independent, no reuse
wasm4 = cloc[
    sturdy_apron('effect/stack/RelationalStack.scala') |
    sturdy_apron('effect/symboltable/RelationalSymbolTable.scala') | 
    sturdy_apron('values/booleans/RelationalBreakIfOps.scala') |
    sturdy_apron('values/convert/.*') |
    sturdy_apron('values/floating/.*')
]
wasm4["language-independent"] = True
wasm4["reuse"] = False

res = pandas.concat([wasm1, wasm2, wasm3, wasm4])

toTex(res)

\def\sumCode{12231 loc\xspace}
\def\sumReuse{9477 loc\xspace}
\def\sumNoReuse{2754 loc\xspace}
\def\sumLanguageIndependent{7891 loc\xspace}
\def\sumLanguageSpecific{4340 loc\xspace}
\def\sumLanguageIndependentReuse{7163 loc\xspace}
\def\sumLanguageIndependentNoReuse{728 loc\xspace}
\def\sumLanguageSpecificReuse{2314 loc\xspace}
\def\sumLanguageSpecificNoReuse{2026 loc\xspace}
\def\sumReuseOrLanguageIndependent{10205 loc\xspace}
\def\percentCode{100.0\%\xspace}
\def\percentReuse{77.5\%\xspace}
\def\percentNoReuse{22.5\%\xspace}
\def\percentLanguageIndependent{64.5\%\xspace}
\def\percentLanguageSpecific{35.5\%\xspace}
\def\percentLanguageIndependentReuse{58.6\%\xspace}
\def\percentLanguageIndependentNoReuse{6.0\%\xspace}
\def\percentLanguageSpecificReuse{18.9\%\xspace}
\def\percentLanguageSpecificNoReuse{16.6\%\xspace}
\def\percentReuseOrLanguageIndependent{83.4\%\xspace}


/tmp/ipykernel_26979/2015922807.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-wasm/src/main/scala/sturdy/language/wasm/'+re, regex= True, na=False)
/tmp/ipykernel_26979/2015922807.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return cloc.filename.str.contains('sturdy-core/src/main/scala/sturdy/'+re, regex= True, na=False)
